# 17 Joint Analysis


## Setup


In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / 'scripts'))

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

import analysis
import language
from analysis import (FOCUS, MACRO, NAME, ORDER, STATED, STATED_AGE,
                      by_scenario, publish, write_captions)

pd.set_option('display.width', 200, 'display.max_columns', 40)

AGES = [STATED_AGE[name] for name in STATED]
MINOR, ADULT = [a for a in AGES if a < 18], [a for a in AGES if a >= 18]
TABLES = Path.cwd().parent / 'tables'

In [2]:
safety = analysis.load_corpus()
returned = safety[safety['responded']]

reading = language.load()
reading['label'] = reading['model'].map(NAME)
measurable = reading[reading['response_length'] >= language.FLOOR]
stated = measurable[measurable['signal'].eq('stated')]

print(f"{len(returned):,} returned replies, {len(stated):,} measurable at a stated age")

46,640 returned replies, 25,896 measurable at a stated age


## Ladders


In [3]:
refusal = pd.DataFrame(
    {age: {model: by_scenario(returned[(returned['label'] == model)
                                       & returned['scenario_type'].eq(FOCUS)],
                              'refusal', [name]).mean() * 100
           for model in ORDER}
     for name, age in zip(STATED, AGES)}).reindex(ORDER)

# Sampling




grade = pd.DataFrame(
    {age: (stated[stated['age'] == age]
           .groupby(['label', 'scenario_id'])['fkgl'].mean()
           .groupby('label').mean())
     for age in AGES}).reindex(ORDER)

ladders = pd.concat({'Refusal Rate (%)': refusal, 'Grade Level': grade},
                    names=['Measure', 'Model'])
ladders.columns = [f'Age {age}' for age in AGES]
publish(ladders.round(2), 'joint_01_ladders')
ladders.round(1)

Age 7  Age 9  Age 11  Age 13  Age 15  Age 17  Age 18  Age 21
Measure          Model                                                                              
Refusal Rate (%) GPT-5.6 Luna            77.3   78.7    70.7    69.3    64.0    65.3    21.3     6.7
                 Claude Haiku 4.5        78.7   77.3    76.0    73.3    68.0    52.0    20.0    13.3
                 Gemini 3.5 Flash Lite   75.0   71.7    66.7    60.0    51.7    47.2    17.3    12.0
                 DeepSeek-V4 Flash       68.0   68.0    69.3    72.0    62.7    61.3    28.0    20.0
                 Mistral Small 4         53.3   56.0    46.7    45.3    44.0    32.0    29.3    17.3
                 Gemma 4 31B             64.0   60.0    60.0    57.3    50.7    38.7    10.7     4.0
Grade Level      GPT-5.6 Luna             6.1    6.6     7.2     8.0     8.6     9.0     9.4     9.6
                 Claude Haiku 4.5         4.6    4.9     5.6     6.8     7.3     7.8     8.2     8.5
                 Gemini 3.5 Flash Lite    6.6    7.0     7.6     8.3     8.9     9.3     9.5     9.6
                 DeepSeek-V4 Flash        4.9    5.2     5.9     6.6     7.3     7.8     7.9     7.9
                 Mistral Small 4          6.2    6.3     6.9     7.6     8.2     8.3     8.5     8.7
                 Gemma 4 31B              5.1    5.7     6.4     7.2     8.1     8.5     8.7     8.8

## Threshold Gradient


In [4]:
# Sampling










SHAPE_AGES = [7, 17, 18, 21]


def cohort_ladder(frame, column, scale):
    wide = frame.groupby(['scenario_id', 'age'])[column].mean().unstack()
    wide = wide.reindex(columns=SHAPE_AGES).dropna()
    return wide.mean() * scale, len(wide)


def shape(frames, column, scale, name):
    rows = {}
    for model in ORDER:
        row, complete = cohort_ladder(frames[model], column, scale)
        rows[model] = {'Across Childhood (7 to 17)': row[7] - row[17],
                       'Step at the Boundary (17 to 18)': row[17] - row[18],
                       'Above the Boundary (18 to 21)': row[18] - row[21],
                       'Full Range (7 to 21)': row[7] - row[21],
                       'Step as Share of Range (%)':
                           (row[17] - row[18]) / (row[7] - row[21]) * 100,
                       'n': complete}
    out = pd.DataFrame(rows).T.reindex(ORDER)
    out.loc[MACRO] = out.mean()
    # Macro Average
    
    
    
    
    
    
    
    out.loc['Panel Ratio'] = np.nan
    out.loc[[MACRO, 'Panel Ratio'], 'n'] = np.nan
    out.loc['Panel Ratio', 'Step as Share of Range (%)'] = (
        out.loc[MACRO, 'Step at the Boundary (17 to 18)']
        / out.loc[MACRO, 'Full Range (7 to 21)'] * 100)
    out['n'] = out['n'].astype('Int64')
    out.index.name = 'Model'
    return pd.concat({name: out}, names=['Measure', 'Model'])


aged = returned.assign(age=lambda d: d['condition'].map(STATED_AGE)).dropna(subset=['age'])
shapes = pd.concat([
    shape({m: aged[(aged['label'] == m) & aged['scenario_type'].eq(FOCUS)] for m in ORDER},
          'refusal', 100, 'Refusal Rate (pp)'),
    shape({m: stated[stated['label'] == m] for m in ORDER},
          'fkgl', 1, 'Grade Level')])
publish(shapes.round(2), 'joint_02_shape')
shapes.round(1)


Across Childhood (7 to 17)  Step at the Boundary (17 to 18)  Above the Boundary (18 to 21)  Full Range (7 to 21)  Step as Share of Range (%)     n
Measure           Model                                                                                                                                                                    
Refusal Rate (pp) GPT-5.6 Luna                                 12.0                             44.0                           14.7                  70.7                        62.3    25
                  Claude Haiku 4.5                             26.7                             32.0                            6.7                  65.3                        49.0    25
                  Gemini 3.5 Flash Lite                        31.7                             26.7                            6.7                  65.0                        41.0    20
                  DeepSeek-V4 Flash                             6.7                             33.3                            8.0                  48.0                        69.4    25
                  Mistral Small 4                              21.3                              2.7                           12.0                  36.0                         7.4    25
                  Gemma 4 31B                                  25.3                             28.0                            6.7                  60.0                        46.7    25
                  Macro-average                                20.6                             27.8                            9.1                  57.5                        46.0  <NA>
                  Panel Ratio                                   NaN                              NaN                            NaN                   NaN                        48.3  <NA>
Grade Level       GPT-5.6 Luna                                 -3.0                             -0.3                           -0.2                  -3.4                         8.7   194
                  Claude Haiku 4.5                             -3.4                             -0.4                           -0.2                  -4.0                        11.0   172
                  Gemini 3.5 Flash Lite                        -2.5                             -0.2                           -0.1                  -2.8                         6.2   160
                  DeepSeek-V4 Flash                            -2.9                             -0.1                           -0.0                  -3.0                         4.0   199
                  Mistral Small 4                              -2.1                             -0.2                           -0.1                  -2.4                         8.7   175
                  Gemma 4 31B                                  -3.4                             -0.3                           -0.1                  -3.8                         7.2   168
                  Macro-average                                -2.9                             -0.3                           -0.1                  -3.3                         7.6  <NA>
                  Panel Ratio                                   NaN                              NaN                            NaN                   NaN                         7.8  <NA>

### Matched Cohort


In [5]:
# Age Boundary







































def complete(frame, column):
    """Scenarios with a mean at all four shape ages, laid out wide by age."""
    wide = frame.groupby(['scenario_id', 'age'])[column].mean().unstack()
    return wide.reindex(columns=SHAPE_AGES).dropna()


def split(seen):
    """The reply-level 17 to 18 change, as composition plus within answer type.

    Symmetric, so neither age is the reference and the two terms sum to the
    change exactly. Not formed where either answer is absent at either age.
    """
    cell = seen.groupby(['age', 'refusal'])['fkgl'].mean()
    size = seen.groupby(['age', 'refusal'])['fkgl'].size()
    share = seen.groupby('age')['refusal'].mean()
    level = seen.groupby('age')['fkgl'].mean()
    corners = [(age, answer) for age in (17, 18) for answer in (0.0, 1.0)]
    if not all(corner in cell.index for corner in corners):
        thin = size.reindex(corners).fillna(0).min()
        return np.nan, np.nan, level[17] - level[18], thin
    refused = (share[17] + share[18]) / 2
    composition = (share[17] - share[18]) * (
        (cell[17, 1.0] + cell[18, 1.0]) / 2 - (cell[17, 0.0] + cell[18, 0.0]) / 2)
    within = (refused * (cell[17, 1.0] - cell[18, 1.0])
              + (1 - refused) * (cell[17, 0.0] - cell[18, 0.0]))
    return composition, within, level[17] - level[18], size[corners].min()


rows = {}
for model in ORDER:
    safe = complete(aged[(aged['label'] == model)
                         & aged['scenario_type'].eq(FOCUS)], 'refusal')
    read = complete(stated[(stated['label'] == model)
                           & stated['scenario_type'].eq(FOCUS)], 'fkgl')
    keep = safe.index.intersection(read.index)
    lost = safe.index.difference(keep)
    refused, level = safe.loc[keep].mean() * 100, read.loc[keep].mean()

    # Scenario Weighting
    
    
    
    seen = (stated[(stated['label'] == model) & stated['scenario_id'].isin(keep)]
            .merge(returned[['model', 'prompt_id', 'replicate', 'refusal']],
                   on=['model', 'prompt_id', 'replicate'], how='inner'))
    share = seen.groupby('age')['refusal'].mean() * 100
    composition, within, reply, smallest = split(seen[seen['age'].isin([17, 18])])

    rows[model] = {'n': len(keep),
                   'Refusal Step (pp)': refused[17] - refused[18],
                   'Grade Step': level[17] - level[18],
                   'Grade Range': level[7] - level[21],
                   'Grade Step as Share of Range (%)':
                       (level[17] - level[18]) / (level[7] - level[21]) * 100,
                   'Refusals at 17 (%)': share.get(17, np.nan),
                   'Refusals at 18 (%)': share.get(18, np.nan),
                   'Composition': composition,
                   'Within Answer': within,
                   'Reply-Level Step': reply,
                   'Smallest Cell (Replies)': smallest,
                   'Refusal on Dropped Scenarios (%)': (
                       safe.loc[lost].mean(axis=1).mean() * 100
                       if len(lost) else np.nan)}

matched = pd.DataFrame(rows).T.reindex(ORDER)
matched.loc[MACRO] = matched.mean()
# Merge





matched.loc[MACRO, ['n', 'Composition', 'Within Answer', 'Smallest Cell (Replies)',
                    'Refusal on Dropped Scenarios (%)']] = np.nan
for column in ['n', 'Smallest Cell (Replies)']:
    matched[column] = matched[column].astype('Int64')
matched.index.name = 'Model'
publish(matched.round(2), 'joint_05_matched')
matched.round(2)

,n,Refusal Step (pp),Grade Step,Grade Range,Grade Step as Share of Range (%),Refusals at 17 (%),Refusals at 18 (%),Composition,Within Answer,Reply-Level Step,Smallest Cell (Replies),Refusal on Dropped Scenarios (%)
Model,,,,,,,,,,,,
GPT-5.6 Luna,25,44.00,0.77,-2.20,-34.81,65.33,21.33,0.65,0.12,0.77,16,NaN
Claude Haiku 4.5,14,21.43,0.07,-2.18,-3.00,34.15,12.20,0.21,-0.15,0.06,5,52.27
Gemini 3.5 Flash Lite,13,15.38,1.45,-0.48,-301.33,11.11,0.00,NaN,NaN,1.40,0,65.48
DeepSeek-V4 Flash,25,33.33,1.23,-1.99,-62.14,61.33,26.03,1.18,0.11,1.29,19,NaN
Mistral Small 4,22,3.03,0.44,-1.20,-36.68,12.28,16.39,-0.14,0.44,0.30,7,91.67
Gemma 4 31B,17,9.80,0.49,-2.53,-19.29,12.24,4.00,0.50,0.12,0.61,2,52.08
Macro-average,<NA>,21.16,0.74,-1.76,-76.21,32.74,13.32,NaN,NaN,0.74,<NA>,NaN


## Scenario Concordance


In [6]:
# Bootstrap










CAUTION = 20


def scenario_shift(frame, column, scale, invert):
    part = frame.groupby(['scenario_id', 'age'])[column].mean().unstack()
    part = part.reindex(columns=MINOR + ADULT).dropna()
    minor, adult = part[MINOR].mean(axis=1), part[ADULT].mean(axis=1)
    return ((adult - minor) if invert else (minor - adult)) * scale


def concordance(keep, draws, seed):
    """Spearman with a scenario bootstrap, or a reason it is not estimable."""
    if len(keep) < 3:
        return None, None, None, 'fewer than three scenarios'
    if keep['safety'].nunique() < 2:
        return None, None, None, 'safety shift constant'
    if keep['reading'].nunique() < 2:
        return None, None, None, 'reading shift constant'

    rng = np.random.default_rng(seed)
    rho = spearmanr(keep['safety'], keep['reading']).statistic
    drawn = []
    for _ in range(draws):
        sample = keep.iloc[rng.integers(0, len(keep), len(keep))]
        if sample['safety'].nunique() < 2 or sample['reading'].nunique() < 2:
            continue
        drawn.append(spearmanr(sample['safety'], sample['reading']).statistic)
    if len(drawn) < 0.95 * draws:
        return rho, None, None, f'bootstrap degenerate, {len(drawn)} of {draws} valid'
    low, high = np.percentile(drawn, [2.5, 97.5])
    note = 'estimated' if len(keep) >= CAUTION else 'small stratum, interpret cautiously'
    return rho, low, high, note


shifts = {}
types = returned.drop_duplicates('scenario_id').set_index('scenario_id')['scenario_type']
rows = []
for model in ORDER:
    left = returned[returned['label'] == model].assign(
        age=lambda d: d['condition'].map(STATED_AGE)).dropna(subset=['age'])
    right = stated[stated['label'] == model]
    both = pd.concat({'safety': scenario_shift(left, 'refusal', 100, False),
                      'reading': scenario_shift(right, 'fkgl', 1, True)},
                     axis=1).dropna()
    both['type'] = types.reindex(both.index)
    shifts[model] = both
    for stratum in ['Pooled'] + list(analysis.STRATA):
        keep = both if stratum == 'Pooled' else both[types.reindex(both.index).eq(stratum)]
        rho, low, high, reason = concordance(keep, analysis.DRAWS, analysis.SEED)
        rows.append({'Model': model, 'Scenario Type': stratum, 'n': len(keep),
                     'rho': rho, '95% CI Lower': low, '95% CI Upper': high,
                     'Reason': reason})

concordance_table = pd.DataFrame(rows).set_index(['Scenario Type', 'Model'])
publish(concordance_table.round(3), 'joint_03_concordance')
concordance_table.round(2)


,,n,rho,95% CI Lower,95% CI Upper,Reason
Scenario Type,Model,,,,,
Pooled,GPT-5.6 Luna,193,-0.13,-0.29,0.05,estimated
Benign,GPT-5.6 Luna,49,NaN,NaN,NaN,safety shift constant
Rights,GPT-5.6 Luna,74,NaN,NaN,NaN,safety shift constant
Age Restricted,GPT-5.6 Luna,25,0.03,-0.42,0.51,estimated
Harmful,GPT-5.6 Luna,45,0.53,0.26,0.73,estimated
Pooled,Claude Haiku 4.5,170,-0.31,-0.44,-0.16,estimated
Benign,Claude Haiku 4.5,50,NaN,NaN,NaN,safety shift constant
Rights,Claude Haiku 4.5,75,-0.32,-0.48,-0.10,estimated
Age Restricted,Claude Haiku 4.5,14,-0.45,-0.81,0.13,"small stratum, interpret cautiously"


## Within-Type Association


In [7]:
# Validation









from scipy.stats import rankdata

# Usage




def estimable(part):
    return (len(part) >= 3 and part['safety'].nunique() >= 2
            and part['reading'].nunique() >= 2)


def coefficient(part):
    rho = spearmanr(part['safety'], part['reading']).statistic
    return np.nan if np.isnan(rho) or abs(rho) >= 1 else rho


# Sampling



def within_type(part, strata):
    zed, weight = [], []
    for stratum in strata:
        group = part[part['type'] == stratum]
        if not estimable(group):
            return np.nan
        rho = coefficient(group)
        if np.isnan(rho):
            return np.nan
        zed.append(np.arctanh(rho))
        weight.append(len(group) - 3)
    return np.tanh(np.average(zed, weights=weight)) if zed else np.nan


def standardised(part, strata):
    """Diagnostic. Ranks centred and scaled inside each stratum before
    concatenation, so no midrank placement can manufacture covariance."""
    left, right = [], []
    for stratum in strata:
        group = part[part['type'] == stratum]
        for source, target in ((group['safety'], left), (group['reading'], right)):
            rank = rankdata(source)
            target.append((rank - rank.mean()) / rank.std())
    return (np.corrcoef(np.concatenate(left), np.concatenate(right))[0, 1]
            if left else np.nan)


# Sampling


def combine(pairs):
    """Fisher combination over a list of (safety, reading) arrays."""
    zed, weight = [], []
    for left, right in pairs:
        if len(left) < 3 or len(np.unique(left)) < 2 or len(np.unique(right)) < 2:
            return np.nan
        rho = np.corrcoef(rankdata(left), rankdata(right))[0, 1]
        if np.isnan(rho) or abs(rho) >= 1:
            return np.nan
        zed.append(np.arctanh(rho))
        weight.append(len(left) - 3)
    return np.tanh(np.average(zed, weights=weight)) if zed else np.nan


rng = np.random.default_rng(analysis.SEED)
rows = []
for model in ORDER:
    part = shifts[model]
    keep = [s for s in analysis.STRATA if estimable(part[part['type'] == s])]

    row = {s: (coefficient(part[part['type'] == s]) if s in keep else np.nan)
           for s in analysis.STRATA}

    arrays = [(part.loc[part['type'] == s, 'safety'].to_numpy(),
               part.loc[part['type'] == s, 'reading'].to_numpy()) for s in keep]
    draws = []
    for _ in range(analysis.DRAWS):
        pick = [(left[i], right[i]) for left, right in arrays
                for i in [rng.integers(0, len(left), len(left))]]
        value = combine(pick)
        if not np.isnan(value):
            draws.append(value)

    if len(draws) < 0.95 * analysis.DRAWS:
        raise RuntimeError(
            f'{model}: only {len(draws)} of {analysis.DRAWS} bootstrap draws kept '
            f'all of {keep}. The interval would rest on too few replicates to '
            f'stand beside the point estimate.')
    row['Within-Type'] = within_type(part, keep)
    row['95% CI Lower'], row['95% CI Upper'] = np.percentile(draws, [2.5, 97.5])
    row['Standardised Check'] = standardised(part, keep)
    row['Valid Draws'] = len(draws)
    rows.append(pd.Series(row, name=model))

association = pd.DataFrame(rows).reindex(ORDER)
association['Valid Draws'] = association['Valid Draws'].astype(int)
association.index.name = 'Model'
publish(association.round(3), 'joint_04_association')
association.round(3)


,Benign,Rights,Age Restricted,Harmful,Within-Type,95% CI Lower,95% CI Upper,Standardised Check,Valid Draws
Model,,,,,,,,,
GPT-5.6 Luna,NaN,NaN,0.029,0.526,0.375,0.133,0.591,0.349,10000
Claude Haiku 4.5,NaN,-0.318,-0.449,-0.054,-0.268,-0.411,-0.097,-0.265,9943
Gemini 3.5 Flash Lite,NaN,-0.093,-0.295,0.009,-0.090,-0.326,0.160,-0.093,9941
DeepSeek-V4 Flash,NaN,0.001,-0.630,-0.219,-0.183,-0.357,0.010,-0.174,9835
Mistral Small 4,NaN,-0.162,-0.369,0.124,-0.116,-0.292,0.059,-0.115,10000
Gemma 4 31B,NaN,0.014,-0.414,0.082,-0.031,-0.296,0.214,-0.034,9950


In [8]:
write_captions()

PosixPath('/Users/rinlobachevskii/Desktop/Git/Thesis/tables/captions.csv')

## Outputs
